# 同時着手将棋 — Colab学習

ランタイム → ランタイムのタイプを変更 → **GPU** を選択。ローカルダッシュボードで保存した `shogi-colab.zip` を用意し、上から実行します。学習はColabの一時ディスクで行い、確定世代をDriveへ退避します。

正式ルールの版は同梱Rustから取得し、上限は200ラウンドです。動作テストと棋力評価を混同しません。Colabの無料枠での速度や棋力向上は未計測です。

In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile, sys, subprocess, shutil, tempfile, json
uploaded = files.upload()  # shogi-colab.zip を選択
assert "shogi-colab.zip" in uploaded, "ローカル画面でソースZIPを保存してください"
PROJECT = Path(tempfile.mkdtemp(prefix="shogi-source-", dir="/content"))
with zipfile.ZipFile("shogi-colab.zip") as z:
    assert sum(i.file_size for i in z.infolist()) < 32 * 1024**2
    for i in z.infolist():
        target = (PROJECT / i.filename).resolve()
        assert target.is_relative_to(PROJECT), "不正なZIPパス"
    z.extractall(PROJECT)
os.chdir(PROJECT)
os.environ["SHOGI_RUNTIME"] = "colab"
print("Source:", PROJECT)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
if not shutil.which("cargo"):
    subprocess.run(["curl", "--proto", "=https", "--tlsv1.2", "-sSf", "https://sh.rustup.rs", "-o", "/tmp/shogi-rustup.sh"], check=True)
    subprocess.run(["sh", "/tmp/shogi-rustup.sh", "-y", "--profile", "minimal", "--default-toolchain", "1.90.0"], check=True)
os.environ["PATH"] = str(Path.home() / ".cargo/bin") + os.pathsep + os.environ["PATH"]
subprocess.run(["cargo", "build", "-p", "shogi-cli", "--release", "--locked"], check=True)
import torch
print("PyTorch", torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    print("GPUランタイムへの切り替えを推奨します。CPUでも動作します。")


## 保存先と実験設定

同じRUN_NAMEは同じソース・設定で再開します。アルゴリズムやルールを変える場合は新しい名前にしてください。Colabは1実行につき1セッションで使用します。設定は最初の学習前に変更してください。既定の8環境・2worker・8候補は開始値で、GPU実測後に調整します。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_NAME = "shogi-efficient-v1"
assert RUN_NAME and all(c.isalnum() or c in "-_" for c in RUN_NAME)
DRIVE_RUN = Path("/content/drive/MyDrive/Simultaneous_Shogi/runs") / RUN_NAME
LOCAL_RUN = Path(tempfile.mkdtemp(prefix="shogi-run-")) / RUN_NAME
if (DRIVE_RUN / "manifest.json").exists():
    shutil.copytree(DRIVE_RUN, LOCAL_RUN, ignore=shutil.ignore_patterns("*.tmp", "RUNNING", "STOP"))
    config = json.loads((LOCAL_RUN / "manifest.json").read_text())["config"]
else:
    config = json.loads((PROJECT / "training/config.json").read_text())
    # 初回だけ変更可能な例: config["generations"] = 20
    # 評価の判定回数を開始前に固定するため、再開時の世代数変更は不可。
CONFIG = Path("/content/shogi-run-config.json")
CONFIG.write_text(json.dumps(config, indent=2))
print("Local scratch:", LOCAL_RUN, "\nDurable mirror:", DRIVE_RUN)
print(json.dumps(config, indent=2))


In [ ]:
command = [sys.executable, "-m", "training.train", "--run-dir", str(LOCAL_RUN), "--config", str(CONFIG), "--mirror-dir", str(DRIVE_RUN)]
if (LOCAL_RUN / "manifest.json").exists():
    command.append("--resume")
# 停止: Colabの実行停止、または別セルで (LOCAL_RUN / "STOP").write_text("stop")
# 進行中の世代は破棄され、最後に確定・退避した世代から再開します。
process = subprocess.Popen(command)
try:
    assert process.wait() == 0, "上のエラーを確認してください"
except KeyboardInterrupt:
    (LOCAL_RUN / "STOP").write_text("requested from notebook")
    process.wait()
print("Last saved generation:", json.loads((LOCAL_RUN / "manifest.json").read_text())["generation"])


## ローカルへ結果を渡す

結果ZIPをローカル画面の「Colab・データ連携」から取り込みます。Drive for desktopを使う場合は、上の `runs` フォルダーを同期して、ローカル画面へ絶対パスを設定します。進行中の世代は確定後に反映します。Driveへ退避する前にランタイムが切断された場合、その世代は再実行されます。

In [ ]:
EXPORT = Path("/content/shogi-results.zip")
subprocess.run([sys.executable, "-m", "training.transfer", "export", "--run-dir", str(LOCAL_RUN), "--archive", str(EXPORT)], check=True)
files.download(str(EXPORT))
